# Exploration des sources de données

Objectif : regarder la structure réelle de chaque dataset (colonnes, langue, taille, exemples) avant d'écrire la logique de conversion dans `scripts/extraction.py`.

In [1]:
import sys
sys.path.append("..")

from datasets import load_dataset
from dotenv import load_dotenv
from scripts.sampling import save_sample

load_dotenv()

/home/rapha/ia-engineer/llm-finetuning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## MediQA (FR)

In [2]:
mediqa = load_dataset("ANR-MALADES/MediQAl", "oeq")
mediqa

DatasetDict({
    test: Dataset({
        features: ['id', 'clinical_case', 'cc_question_number', 'question', 'answer', 'medical_subject', 'question_type'],
        num_rows: 4969
    })
})

In [3]:
split = list(mediqa.keys())[0]
len(mediqa[split])

4969

In [4]:
mediqa[split].features

{'id': Value('string'),
 'clinical_case': Value('string'),
 'cc_question_number': Value('string'),
 'question': Value('string'),
 'answer': Value('string'),
 'medical_subject': Value('string'),
 'question_type': Value('string')}

In [5]:
mediqa[split][:5]

{'id': ['1', '2', '3', '4', '5'],
 'clinical_case': ['Homme, 58 ans, majoration de dyspnée chez un BPCO connu depuis 20 ans.\n\n Examen d’entrée : \n Constantes :\n - FC = 130bpm\n - PA = 130/80 aux deux bras\n - FR = 40/min\n - SpO2 = 80%\n - T = 38 ,5°C\n Tirage sus-sternal\n Cyanose\n Sueurs\n Auscultation : \n - Ronchi bilatéraux et sibilants\n - Tympanisme bilatéral\n Hépatomégalie douloureuse\n Reflux hépato-jugulaire\n Turgescence jugulaire\n Hippocratisme digital\n Cyanose unguéale\n \n ATCD \n Fumeur (80 PA)\n BPCO\n \n Traitement habituel\n β2 mimétiques \n \n HDM : depuis 1 semaine : ne peut plus fumer, expectorations sales \n \n 1h après, malgré le traitement initial, patient tient des propos incohérents et désorienté. ',
  'Homme, 58 ans, majoration de dyspnée chez un BPCO connu depuis 20 ans.\n\n Examen d’entrée : \n Constantes :\n - FC = 130bpm\n - PA = 130/80 aux deux bras\n - FR = 40/min\n - SpO2 = 80%\n - T = 38 ,5°C\n Tirage sus-sternal\n Cyanose\n Sueurs\n Auscultat

MediQAl propose trois configs : `oeq` (questions ouvertes, réponse rédigée librement), `mcqm` (QCM à réponses multiples) et `mcqu` (QCM à réponse unique).

Pour l'instant je charge seulement `oeq`. Raison : c'est le format le plus proche de ce qu'on veut au final, un agent qui répond en langage libre plutôt qu'un agent qui coche des cases, et FrenchMedMCQA couvre déjà le format QCM côté français. Si le volume `oeq` ne suffit pas pour le SFT, `mcqm` et `mcqu` restent disponibles en complément.

Un seul split (`test`), 4969 exemples. Chaque exemple contient un cas clinique, une question posée sur ce cas et une réponse rédigée, avec un sujet médical et un type de question en plus. Bon candidat pour le SFT.

In [6]:
save_sample(mediqa, "mediqa")

mediqa/test : 20 exemples écrits dans /home/rapha/ia-engineer/llm-finetuning/data/samples/mediqa/test.json


## FrenchMedMCQA

In [7]:
frenchmedmcqa = load_dataset("nthngdy/frenchmedmcqa")
frenchmedmcqa

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'answer_a', 'answer_b', 'answer_c', 'answer_d', 'answer_e', 'correct_answers', 'number_correct_answers'],
        num_rows: 595
    })
    validation: Dataset({
        features: ['id', 'question', 'answer_a', 'answer_b', 'answer_c', 'answer_d', 'answer_e', 'correct_answers', 'number_correct_answers'],
        num_rows: 164
    })
    test: Dataset({
        features: ['id', 'question', 'answer_a', 'answer_b', 'answer_c', 'answer_d', 'answer_e', 'correct_answers', 'number_correct_answers'],
        num_rows: 321
    })
})

In [8]:
split = list(frenchmedmcqa.keys())[0]
len(frenchmedmcqa[split])

595

In [9]:
frenchmedmcqa[split].features

{'id': Value('string'),
 'question': Value('string'),
 'answer_a': Value('string'),
 'answer_b': Value('string'),
 'answer_c': Value('string'),
 'answer_d': Value('string'),
 'answer_e': Value('string'),
 'correct_answers': Value('int64'),
 'number_correct_answers': ClassLabel(names=['1', '2', '3', '4', '5'])}

In [10]:
frenchmedmcqa[split][:5]

{'id': ['230bac49b0fe863b772410bc8d01a025f63c3c999065480131d6334abd2efeff',
  '0ca718ff033bb68503e41c4d30b4f7580cb3f220e000a2e2d8520803ffdb8b06',
  '3e292fc26ab1844f7d3805a5cceba0534261fcd65446dc544b4f2f7e3459de01',
  '6c35039202449b2b89e7b8a79dd0d7ae1b73ad650988446f331d3a26e8ecff60',
  'dd39598f83a51258dc16cd37c21496ed298cf66b7245f80217a5fa9891c509ac'],
 'question': ['Parmi les affirmations suivantes, une seule est fausse, indiquer laquelle: les particules alpha',
  "Parmi les bactéries suivantes, une seule ne peut généralement pas être responsable d'une méningite aiguë, laquelle?",
  'Parmi les propositions suivantes, indiquer celle qui est exacte. Le crack est une forme:',
  'Parmi les propositions suivantes, une seule est exacte. Laquelle? La sérotonine est le (la)\xa0:',
  'Parmi les techniques voltampérométriques, on trouve:'],
 'answer_a': ["Sont formées de noyaux d'hélium",
  'Haemophilus influenzae',
  "D'héroine",
  '5-hydroxy tryptophane',
  'La potentiométrie'],
 'answer_b'

Trois splits (train 595, validation 164, test 321). QCM à 5 propositions, avec parfois plusieurs bonnes réponses (`number_correct_answers`). Il faudra reformuler ça en format question/réponse pour le SFT.

In [11]:
save_sample(frenchmedmcqa, "frenchmedmcqa")

frenchmedmcqa/train : 20 exemples écrits dans /home/rapha/ia-engineer/llm-finetuning/data/samples/frenchmedmcqa/train.json
frenchmedmcqa/validation : 20 exemples écrits dans /home/rapha/ia-engineer/llm-finetuning/data/samples/frenchmedmcqa/validation.json
frenchmedmcqa/test : 20 exemples écrits dans /home/rapha/ia-engineer/llm-finetuning/data/samples/frenchmedmcqa/test.json


## MedQuAD (EN)

In [12]:
medquad = load_dataset("keivalya/MedQuad-MedicalQnADataset")
medquad

DatasetDict({
    train: Dataset({
        features: ['qtype', 'Question', 'Answer'],
        num_rows: 16407
    })
})

In [13]:
split = list(medquad.keys())[0]
len(medquad[split])

16407

In [14]:
medquad[split].features

{'qtype': Value('string'),
 'Question': Value('string'),
 'Answer': Value('string')}

In [15]:
medquad[split][:5]

{'qtype': ['susceptibility',
  'symptoms',
  'susceptibility',
  'exams and tests',
  'treatment'],
 'Question': ['Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?',
  'What are the symptoms of Lymphocytic Choriomeningitis (LCM) ?',
  'Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?',
  'How to diagnose Lymphocytic Choriomeningitis (LCM) ?',
  'What are the treatments for Lymphocytic Choriomeningitis (LCM) ?'],
 'Answer': ['LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected rodents.  Transmission may also occur when these materials are directly introduced into broken skin, the nose, the eyes, or the mouth, or presumably, via the bite of an infected rodent. Person-to-person transmission has not been reported, with the exception of vertical transmission from infected mother to fetus, and rarely, through organ transplantation.',
  'LCMV is most commonly recognized as causing neurological disease, as its name impl

Un seul split (`train`), 16407 exemples, en anglais. Structure simple : question, réponse et un type de question (`qtype`). À voir si on traduit ou si on filtre selon la langue cible du projet.

In [16]:
save_sample(medquad, "medquad")

medquad/train : 20 exemples écrits dans /home/rapha/ia-engineer/llm-finetuning/data/samples/medquad/train.json


## UltraMedical-Preference (EN, pour le DPO)

In [17]:
ultramedical_preference = load_dataset("TsinghuaC3I/UltraMedical-Preference")
ultramedical_preference

DatasetDict({
    train: Dataset({
        features: ['prompt_id', 'label_type', 'prompt', 'chosen', 'rejected', 'metadata', 'feedback'],
        num_rows: 109353
    })
    validation: Dataset({
        features: ['prompt_id', 'label_type', 'prompt', 'chosen', 'rejected', 'metadata', 'feedback'],
        num_rows: 2232
    })
    test: Dataset({
        features: ['prompt_id', 'label_type', 'prompt', 'chosen', 'rejected', 'metadata', 'feedback'],
        num_rows: 777
    })
})

In [18]:
split = list(ultramedical_preference.keys())[0]
len(ultramedical_preference[split])

109353

In [19]:
ultramedical_preference[split].features

{'prompt_id': Value('string'),
 'label_type': Value('string'),
 'prompt': Value('string'),
 'chosen': List({'content': Value('string'), 'role': Value('string')}),
 'rejected': List({'content': Value('string'), 'role': Value('string')}),
 'metadata': {'golden_answer': Value('string'),
  'chosen': {'model': Value('string'),
   'score': Value('float64'),
   'rank': Value('int64'),
   'evaluation': Value('string')},
  'rejected': {'model': Value('string'),
   'score': Value('float64'),
   'rank': Value('int64'),
   'evaluation': Value('string')}},
 'feedback': Value('string')}

In [20]:
ultramedical_preference[split][:5]

{'prompt_id': ['WikiInstruct,8304',
  'WikiInstruct,8846',
  'MedMCQA,11404',
  'ChatDoctor,33567',
  'WikiInstruct,14624'],
 'label_type': ['length', 'length', 'length', 'easy', 'length'],
 'prompt': ['Investigate the intricacies of immunometabolism, a distinct subfield of immunology that examines the interconnection between cellular metabolic processes and the functional attributes of immune cells. Clarify the mechanisms by which this symbiosis modulates the comprehensive immune response, and analyze its integration within the broader spectrum of immunological studies, with an emphasis on the implications for metabolic diseases and the consequential effects on the proficiency of the immune system.',
  'Examine the differential immunotoxic effects exerted by distinct microplastic polymer varieties on the innate and adaptive immune mechanisms across a range of marine organisms, emphasizing the plausible perturbations within their intracellular communication networks, and delineate the 

Trois splits (train 109353, validation 2232, test 777). Langue : anglais, prompts et réponses inclus. Chaque exemple a un prompt, une réponse `chosen` et une réponse `rejected`, avec des métadonnées de notation par modèle (score, rang, évaluation). C'est le format attendu pour le DPO.

Point à trancher : le SFT se fait en français (MediQAl, FrenchMedMCQA) alors que ce dataset DPO est entièrement en anglais. Il faudra décider si on traduit ce dataset, si on trouve une alternative en français, ou si on assume un DPO en anglais sur un modèle fine tuné en français.

In [21]:
save_sample(ultramedical_preference, "ultramedical_preference")

ultramedical_preference/train : 20 exemples écrits dans /home/rapha/ia-engineer/llm-finetuning/data/samples/ultramedical_preference/train.json
ultramedical_preference/validation : 20 exemples écrits dans /home/rapha/ia-engineer/llm-finetuning/data/samples/ultramedical_preference/validation.json
ultramedical_preference/test : 20 exemples écrits dans /home/rapha/ia-engineer/llm-finetuning/data/samples/ultramedical_preference/test.json


## Synthèse

Quatre datasets explorés, à combiner en un schéma commun avant d'écrire la logique dans `scripts/extraction.py`.

Pour le SFT (français) :
- MediQAl (`oeq`) : 4969 exemples, cas clinique + question + réponse rédigée, split `test` uniquement
- FrenchMedMCQA : 595 / 164 / 321 (train / validation / test), QCM à 5 propositions, à reformuler en question/réponse
- MedQuAD : 16407 exemples en anglais, split `train` uniquement

Pour le DPO :
- UltraMedical-Preference : 109353 / 2232 / 777 (train / validation / test), en anglais, format prompt / chosen / rejected déjà prêt

Points à trancher avant l'extraction :
- Langue : le SFT visé est en français mais MedQuAD et UltraMedical-Preference sont en anglais. Il faut décider entre traduction, filtrage, ou DPO en anglais sur un modèle fine tuné en français.
- MediQAl et MedQuAD n'ont pas de split train/test, il faudra en créer un nous mêmes pour l'évaluation.
- `number_correct_answers` dans FrenchMedMCQA vaut 0 partout dans l'aperçu, à vérifier avant d'écrire la conversion (probable souci d'encodage du ClassLabel).

Des échantillons de chaque split sont sauvegardés dans `data/samples/` via `scripts/sampling.py`, pour relire la structure sans recharger les datasets.